In [ ]:
import os
import random
import numpy as np
import pandas as pd
import segyio
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Data Cleaning

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================

BASE_PATH = "../data/BHOOMATHON_data"

WELL17 = f"{BASE_PATH}/Wells/well17_lab_calibrated.xlsx"
WELL22 = f"{BASE_PATH}/Wells/well22_lab_calibrated.xlsx"

# =========================================================
# LOAD FILES
# =========================================================

df17 = pd.read_excel(WELL17)
df22 = pd.read_excel(WELL22)

# =========================================================
# STEP 1:
# CONVERT ALL COLUMN NAMES TO LOWERCASE
# =========================================================

df17.columns = df17.columns.str.lower()
df22.columns = df22.columns.str.lower()

print("\n===== WELL17 COLUMNS =====")
print(df17.columns)

print("\n===== WELL22 COLUMNS =====")
print(df22.columns)

# =========================================================
# STEP 2:
# CALCULATE GCV
#
# Formula:
# GCV = 85.6 * [100 - (1.1*A + M)] - 60*M
#
# A = Ash
# M = Moisture
# =========================================================

df17["gcv"] = (
    85.6 * (100 - (1.1 * df17["ash"] + df17["moisture"]))
    - 60 * df17["moisture"]
)

df22["gcv"] = (
    85.6 * (100 - (1.1 * df22["ash"] + df22["moisture"]))
    - 60 * df22["moisture"]
)

# =========================================================
# SAVE UPDATED FILES
# =========================================================

OUT17 = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"
OUT22 = f"{BASE_PATH}/Wells/well22_lab_calibrated_gcv.xlsx"

df17.to_excel(OUT17, index=False)
df22.to_excel(OUT22, index=False)

# =========================================================
# DISPLAY RESULTS
# =========================================================

print("\n===== WELL17 SAMPLE =====")
print(df17.head())

print("\n===== WELL22 SAMPLE =====")
print(df22.head())

print("\n===================================")
print("GCV calculation completed successfully")
print("Saved files:")
print(OUT17)
print(OUT22)
print("===================================")

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================

BASE_PATH = "../data/BHOOMATHON_data"

FILE17 = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"
FILE22 = f"{BASE_PATH}/Wells/well22_lab_calibrated_gcv.xlsx"

# =========================================================
# LOAD FILES
# =========================================================

df17 = pd.read_excel(FILE17)
df22 = pd.read_excel(FILE22)

# =========================================================
# FUNCTION TO CHECK NaN VALUES
# =========================================================

def nan_report(df, name):

    print(f"\n================ {name} ================\n")

    # Total rows having at least one NaN
    rows_with_nan = df.isnull().any(axis=1).sum()

    # Total columns having at least one NaN
    cols_with_nan = df.isnull().any(axis=0).sum()

    print(f"Total Rows            : {len(df)}")
    print(f"Total Columns         : {len(df.columns)}")
    print(f"Rows containing NaN   : {rows_with_nan}")
    print(f"Columns containing NaN: {cols_with_nan}")

    print("\n========== NaN Count Per Column ==========\n")

    nan_per_column = df.isnull().sum()

    # Show only columns having NaN
    nan_per_column = nan_per_column[nan_per_column > 0]

    if len(nan_per_column) == 0:
        print("No NaN values found.")
    else:
        print(nan_per_column.sort_values(ascending=False))

# =========================================================
# RUN REPORTS
# =========================================================

nan_report(df17, "WELL17")
nan_report(df22, "WELL22")

In [ ]:
import pandas as pd

# =========================================================
# PATHS
# =========================================================

BASE_PATH = "../data/BHOOMATHON_data"

FILE17 = f"{BASE_PATH}/Wells/well17_lab_calibrated_gcv.xlsx"
FILE22 = f"{BASE_PATH}/Wells/well22_lab_calibrated_gcv.xlsx"

# =========================================================
# LOAD FILES
# =========================================================

df17 = pd.read_excel(FILE17)
df22 = pd.read_excel(FILE22)

# =========================================================
# HANDLE NaN VALUES IN COALSEAM COLUMN
# =========================================================

df17["coalseam"] = df17["coalseam"].fillna("No Available")
df22["coalseam"] = df22["coalseam"].fillna("No Available")

# =========================================================
# CHECK AGAIN
# =========================================================

print("\n===== WELL17 NaN COUNT =====")
print(df17.isnull().sum())

print("\n===== WELL22 NaN COUNT =====")
print(df22.isnull().sum())

# =========================================================
# SAVE UPDATED FILES
# =========================================================

OUT17 = f"{BASE_PATH}/Wells/well17_final_cleaned.xlsx"
OUT22 = f"{BASE_PATH}/Wells/well22_final_cleaned.xlsx"

df17.to_excel(OUT17, index=False)
df22.to_excel(OUT22, index=False)

print("\n===================================")
print("NaN handling completed successfully")
print("Saved cleaned files:")
print(OUT17)
print(OUT22)
print("===================================")

# ============================================================
# DATASET
# ============================================================

In [ ]:


class MultiModalDataset(Dataset):

    def __init__(
        self,
        logs,
        seismic,
        targets
    ):

        self.logs = torch.tensor(
            logs,
            dtype=torch.float32
        )

        self.seismic = torch.tensor(
            seismic,
            dtype=torch.float32
        )

        self.targets = torch.tensor(
            targets,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.logs)

    def __getitem__(self, idx):

        return (
            self.logs[idx],
            self.seismic[idx],
            self.targets[idx]
        )

train_dataset = MultiModalDataset(
    X_train_logs,
    X_train_seis,
    y_train
)

test_dataset = MultiModalDataset(
    X_test_logs,
    X_test_seis,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)


In [ ]:
import os
import numpy as np
import pandas as pd
import segyio
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# ============================================================
# PATHS
# ============================================================

BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

WELL17 = f"{BASE_PATH}/Wells/well17_final_cleaned.xlsx"
WELL22 = f"{BASE_PATH}/Wells/well22_final_cleaned.xlsx"

# ============================================================
# LOAD ALL SEISMIC FILES
# ============================================================

print("\n============================================================")
print("LOADING ALL SEISMIC FILES")
print("============================================================")

seismic_cubes = []

for path in SEISMIC_FILES:

    print(f"\nLoading: {os.path.basename(path)}")

    with segyio.open(path, "r", ignore_geometry=True) as f:
        cube = np.stack([np.copy(x) for x in f.trace[:]])

    print("Original Shape:", cube.shape)

    # Normalize each cube independently
    cube = (cube - cube.mean()) / (cube.std() + 1e-8)

    seismic_cubes.append(cube)

print("\nTotal Seismic Cubes Loaded:", len(seismic_cubes))

# ============================================================
# LOAD WELL DATA
# ============================================================

print("\n============================================================")
print("LOADING WELL DATA")
print("============================================================")

well17 = pd.read_excel(WELL17)
well22 = pd.read_excel(WELL22)

# TRAIN ON WELL17
# TEST ON WELL22

train_df = well17.copy()
test_df = well22.copy()

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

# ============================================================
# FEATURES
# ============================================================

FEATURE_COLUMNS = [
    'caliper',
    'density',
    'natural_gamma',
    'resistivity',
    'p_slowness',
    's_slowness',
    'p_velocity',
    's_velocity',
    'ash',
    'moisture'
]

TARGET_COLUMN = 'gcv'

# ============================================================
# REMOVE NaN
# ============================================================

train_df = train_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])
test_df = test_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])

# ============================================================
# NORMALIZATION
# ============================================================

scaler_x = StandardScaler()

X_train_logs = scaler_x.fit_transform(train_df[FEATURE_COLUMNS])
X_test_logs = scaler_x.transform(test_df[FEATURE_COLUMNS])

scaler_y = StandardScaler()

y_train = scaler_y.fit_transform(
    train_df[[TARGET_COLUMN]]
).flatten()

y_test = scaler_y.transform(
    test_df[[TARGET_COLUMN]]
).flatten()

# ============================================================
# CREATE MULTI-SEISMIC PATCHES
# ============================================================

PATCH_H = 64
PATCH_W = 128

def create_multichannel_patches(df):

    patches = []

    for i in range(len(df)):

        multi_channel_patch = []

        for cube in seismic_cubes:

            # pseudo mapping
            trace_idx = int(i % cube.shape[0])
            sample_idx = int((i * 20) % cube.shape[1])

            h1 = max(0, trace_idx - PATCH_H // 2)
            h2 = min(cube.shape[0], trace_idx + PATCH_H // 2)

            w1 = max(0, sample_idx - PATCH_W // 2)
            w2 = min(cube.shape[1], sample_idx + PATCH_W // 2)

            patch = cube[h1:h2, w1:w2]

            padded = np.zeros((PATCH_H, PATCH_W))

            padded[:patch.shape[0], :patch.shape[1]] = patch

            multi_channel_patch.append(padded)

        multi_channel_patch = np.stack(multi_channel_patch)

        patches.append(multi_channel_patch)

    return np.array(patches)

print("\n============================================================")
print("CREATING TRAIN PATCHES")
print("============================================================")

X_train_seis = create_multichannel_patches(train_df)

print("Train Patch Shape:", X_train_seis.shape)

print("\n============================================================")
print("CREATING TEST PATCHES")
print("============================================================")

X_test_seis = create_multichannel_patches(test_df)

print("Test Patch Shape:", X_test_seis.shape)

# ============================================================
# DATASET
# ============================================================

class MultiModalDataset(Dataset):

    def __init__(self, logs, seismic, targets):

        self.logs = torch.tensor(logs, dtype=torch.float32)
        self.seismic = torch.tensor(seismic, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.logs)

    def __getitem__(self, idx):

        return (
            self.logs[idx],
            self.seismic[idx],
            self.targets[idx]
        )

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    drop_last=True
)

test_dataset = MultiModalDataset(
    X_test_logs,
    X_test_seis,
    y_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16
)

# ============================================================
# IMPROVED SEISMIC CNN
# ============================================================

class SeismicCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.cnn = nn.Sequential(

            nn.Conv2d(4, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((4,4))
        )

        self.fc = nn.Sequential(

            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128)
        )

    def forward(self, x):

        x = self.cnn(x)

        x = x.view(x.size(0), -1)

        x = self.fc(x)

        return x

# ============================================================
# LOG MLP
# ============================================================

class LogMLP(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.ReLU()
        )

    def forward(self, x):

        return self.net(x)

# ============================================================
# FUSION MODEL
# ============================================================

class FusionModel(nn.Module):

    def __init__(self, log_dim):

        super().__init__()

        self.seismic_branch = SeismicCNN()

        self.log_branch = LogMLP(log_dim)

        self.fusion = nn.Sequential(

            nn.Linear(128 + 64, 256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, logs, seismic):

        seis_feat = self.seismic_branch(seismic)

        log_feat = self.log_branch(logs)

        combined = torch.cat(
            [seis_feat, log_feat],
            dim=1
        )

        out = self.fusion(combined)

        return out.squeeze()

# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nUsing Device:", device)

# ============================================================
# MODEL
# ============================================================

model = FusionModel(
    len(FEATURE_COLUMNS)
).to(device)

criterion = nn.HuberLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0003,
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)



In [ ]:
train_losses = []
test_losses = []

In [ ]:
EPOCHS = 100
best_loss = float('inf')
for epoch in range(EPOCHS):

    # ======================
    # TRAIN
    # ======================
    model.train()
    running_loss = 0

    for logs, seismic_batch, targets in train_loader:

        logs = logs.to(device)
        seismic_batch = seismic_batch.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(logs, seismic_batch)

        loss = criterion(outputs, targets)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    # ======================
    # TEST (VALIDATION)
    # ======================
    model.eval()
    test_running_loss = 0

    with torch.no_grad():
        for logs, seismic_batch, targets in test_loader:

            logs = logs.to(device)
            seismic_batch = seismic_batch.to(device)
            targets = targets.to(device)

            outputs = model(logs, seismic_batch)

            loss = criterion(outputs, targets)

            test_running_loss += loss.item()

    test_loss = test_running_loss / len(test_loader)

    # ======================
    # LOGGING
    # ======================
    train_losses.append(train_loss)
    test_losses.append(test_loss)

    scheduler.step(test_loss)

    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    # SAVE BEST MODEL (based on TEST loss, not train)
    if test_loss < best_loss:
        best_loss = test_loss
        torch.save(model.state_dict(), "best_multimodal_model.pth")

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Test Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Test Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
preds = []
truths = []

model.eval()

with torch.no_grad():
    for logs, seismic_batch, targets in test_loader:

        logs = logs.to(device)
        seismic_batch = seismic_batch.to(device)

        outputs = model(logs, seismic_batch)

        preds.extend(outputs.cpu().numpy())
        truths.extend(targets.numpy())

preds = np.array(preds)
truths = np.array(truths)

# inverse transform
preds_real = scaler_y.inverse_transform(
    preds.reshape(-1,1)
).flatten()

truths_real = scaler_y.inverse_transform(
    truths.reshape(-1,1)
).flatten()

In [ ]:
plt.figure(figsize=(6,10))

depth = np.linspace(0, 600, len(preds_real))

# TRUE (orange)
plt.scatter(
    truths_real,
    depth,
    color='orange',
    s=15,
    label='True GCV'
)

# PREDICTED (blue)
plt.scatter(
    preds_real,
    depth,
    color='blue',
    s=15,
    label='Predicted GCV'
)



# invert depth (important)
plt.gca().invert_yaxis()

plt.xlabel("GCV")
plt.ylabel("Depth (m)")
plt.title("GCV vs Depth (0–600 m)")

plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
train_losses = []
test_losses = []

train_mae_list = []
test_mae_list = []

train_rmse_list = []
test_rmse_list = []

train_r2_list = []
test_r2_list = []

In [ ]:
import os
import numpy as np
import pandas as pd
import segyio
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

# ============================================================
# PATHS
# ============================================================

BASE_PATH = "../data/BHOOMATHON_data"

SEISMIC_FILES = [
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSDM/Final_PSDMStack_TS_44.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TD_39.sgy",
    f"{BASE_PATH}/Seismic/PSTM/Final_PSTMStack_TS_44.sgy"
]

WELL17 = f"{BASE_PATH}/Wells/well17_final_cleaned.xlsx"
WELL22 = f"{BASE_PATH}/Wells/well22_final_cleaned.xlsx"

# ============================================================
# LOAD SEISMIC
# ============================================================

seismic_cubes = []
for path in SEISMIC_FILES:
    with segyio.open(path, "r", ignore_geometry=True) as f:
        cube = np.stack([np.copy(x) for x in f.trace[:]])
    cube = (cube - cube.mean()) / (cube.std() + 1e-8)
    seismic_cubes.append(cube)

# ============================================================
# LOAD WELLS
# ============================================================

train_df = pd.read_excel(WELL17)
test_df = pd.read_excel(WELL22)

FEATURE_COLUMNS = [
    'caliper','density','natural_gamma','resistivity',
    'p_slowness','s_slowness','p_velocity','s_velocity',
    'ash','moisture'
]
TARGET_COLUMN = 'gcv'

train_df = train_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])
test_df = test_df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN])

# ============================================================
# NORMALIZATION
# ============================================================

scaler_x = StandardScaler()
X_train_logs = scaler_x.fit_transform(train_df[FEATURE_COLUMNS])
X_test_logs = scaler_x.transform(test_df[FEATURE_COLUMNS])

scaler_y = StandardScaler()
y_train = scaler_y.fit_transform(train_df[[TARGET_COLUMN]]).flatten()
y_test = scaler_y.transform(test_df[[TARGET_COLUMN]]).flatten()

# ============================================================
# PATCHES
# ============================================================

PATCH_H, PATCH_W = 64, 128

def create_patches(df):
    patches = []
    for i in range(len(df)):
        multi = []
        for cube in seismic_cubes:
            ti = i % cube.shape[0]
            si = (i*20) % cube.shape[1]

            patch = cube[
                max(0, ti-PATCH_H//2):ti+PATCH_H//2,
                max(0, si-PATCH_W//2):si+PATCH_W//2
            ]

            pad = np.zeros((PATCH_H, PATCH_W))
            pad[:patch.shape[0], :patch.shape[1]] = patch
            multi.append(pad)

        patches.append(np.stack(multi))
    return np.array(patches)

X_train_seis = create_patches(train_df)
X_test_seis = create_patches(test_df)

# ============================================================
# DATASET
# ============================================================

class MultiModalDataset(Dataset):
    def __init__(self, logs, seismic, targets):
        self.logs = torch.tensor(logs, dtype=torch.float32)
        self.seismic = torch.tensor(seismic, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.logs)

    def __getitem__(self, idx):
        return self.logs[idx], self.seismic[idx], self.targets[idx]

train_loader = DataLoader(MultiModalDataset(X_train_logs, X_train_seis, y_train), batch_size=16, shuffle=True)
test_loader = DataLoader(MultiModalDataset(X_test_logs, X_test_seis, y_test), batch_size=16)

# ============================================================
# MODEL
# ============================================================

class FusionModel(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.seis = nn.Sequential(
            nn.Conv2d(4,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4))
        )
        self.seis_fc = nn.Linear(128*4*4,128)

        self.logs = nn.Sequential(
            nn.Linear(dim,128), nn.ReLU(),
            nn.Linear(128,64), nn.ReLU()
        )

        self.fusion = nn.Sequential(
            nn.Linear(128+64,128), nn.ReLU(),
            nn.Linear(128,1)
        )

    def forward(self, logs, seis):
        s = self.seis(seis)
        s = s.view(s.size(0), -1)
        s = self.seis_fc(s)

        l = self.logs(logs)

        return self.fusion(torch.cat([s,l],1)).view(-1)

# ============================================================
# TRAIN SETUP
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = FusionModel(len(FEATURE_COLUMNS)).to(device)
criterion = nn.HuberLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# ============================================================
# TRAIN LOOP
# ============================================================

EPOCHS = 100
best_loss = float('inf')

train_rmse_list, test_rmse_list = [], []
train_mae_list, test_mae_list = [], []
train_r2_list, test_r2_list = [], []

for epoch in range(EPOCHS):

    model.train()
    tr_p, tr_t = [], []

    for logs,seis,t in train_loader:
        logs,seis,t = logs.to(device),seis.to(device),t.to(device)

        optimizer.zero_grad()
        out = model(logs,seis)
        loss = criterion(out,t)
        loss.backward()
        optimizer.step()

        tr_p.extend(out.detach().cpu().numpy())
        tr_t.extend(t.cpu().numpy())

    model.eval()
    te_p, te_t = [], []

    with torch.no_grad():
        for logs,seis,t in test_loader:
            logs,seis,t = logs.to(device),seis.to(device),t.to(device)

            out = model(logs,seis)
            te_p.extend(out.cpu().numpy())
            te_t.extend(t.cpu().numpy())

    # inverse
    tr_p = scaler_y.inverse_transform(np.array(tr_p).reshape(-1,1)).flatten()
    tr_t = scaler_y.inverse_transform(np.array(tr_t).reshape(-1,1)).flatten()
    te_p = scaler_y.inverse_transform(np.array(te_p).reshape(-1,1)).flatten()
    te_t = scaler_y.inverse_transform(np.array(te_t).reshape(-1,1)).flatten()

    # metrics
    train_rmse = np.sqrt(mean_squared_error(tr_t, tr_p))
    test_rmse = np.sqrt(mean_squared_error(te_t, te_p))

    train_mae = mean_absolute_error(tr_t, tr_p)
    test_mae = mean_absolute_error(te_t, te_p)

    train_r2 = r2_score(tr_t, tr_p)
    test_r2 = r2_score(te_t, te_p)

    # store
    train_rmse_list.append(train_rmse)
    test_rmse_list.append(test_rmse)

    train_mae_list.append(train_mae)
    test_mae_list.append(test_mae)

    train_r2_list.append(train_r2)
    test_r2_list.append(test_r2)

    print(f"Epoch {epoch+1} | RMSE {test_rmse:.2f} | MAE {test_mae:.2f} | R2 {test_r2:.3f}")

# ============================================================
# PLOT ALL THREE
# ============================================================

plt.figure(figsize=(18,5))

plt.subplot(1,3,1)
plt.plot(train_rmse_list,label="Train")
plt.plot(test_rmse_list,label="Test")
plt.title("RMSE")

plt.subplot(1,3,2)
plt.plot(train_mae_list,label="Train")
plt.plot(test_mae_list,label="Test")
plt.title("MAE")

plt.subplot(1,3,3)
plt.plot(train_r2_list,label="Train")
plt.plot(test_r2_list,label="Test")
plt.title("R2")

plt.legend()
plt.show()

# ============================================================
# DEPTH PLOT
# ============================================================

depth = np.linspace(0,600,len(te_p))

plt.figure(figsize=(6,10))
plt.scatter(te_t, depth, color='orange', s=10, label='True')
plt.scatter(te_p, depth, color='blue', s=10, label='Pred')
plt.gca().invert_yaxis()
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_mae_list, label="Train MAE")
#plt.plot(test_mae_list, label="Test MAE")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("MAE vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_rmse_list, label="Train RMSE")
#plt.plot(test_rmse_list, label="Test RMSE")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.title("RMSE vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(train_r2_list, label="Train R2")
#plt.plot(test_r2_list, label="Test R2")
plt.xlabel("Epoch")
plt.ylabel("R2 Score")
plt.title("R2 vs Epoch")
plt.legend()
plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_mae_list, label="Train MAE")
plt.plot(test_mae_list, label="Test MAE")

plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.title("MAE vs Epoch")

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(18,5))

# ================= RMSE =================
plt.subplot(1,3,1)
plt.plot(train_rmse_list, label="Train", linewidth=2)
plt.plot(test_rmse_list, label="Test", linewidth=2)
plt.title("RMSE vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("RMSE")
plt.legend()
plt.grid(alpha=0.3)

# ================= MAE =================
plt.subplot(1,3,2)
plt.plot(train_mae_list, label="Train", linewidth=2)
plt.plot(test_mae_list, label="Test", linewidth=2)
plt.title("MAE vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.legend()
plt.grid(alpha=0.3)

# ================= R2 =================
plt.subplot(1,3,3)
plt.plot(train_r2_list, label="Train", linewidth=2)
plt.plot(test_r2_list, label="Test", linewidth=2)
plt.title("R² vs Epoch")
plt.xlabel("Epoch")
plt.ylabel("R²")
plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# GCV Quaility prediction

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../results/coal_quality_predictions.csv")

x = range(len(df))  # index (or depth if available)

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(x, df["TRUE_MOISTURE"], label="True Moisture", color="orange", linewidth=2)
plt.plot(x, df["PRED_MOISTURE"], label="Pred Moisture", color="blue", linewidth=2)

plt.xlabel("Sample Index")
plt.ylabel("Moisture")
plt.title("MOISTURE: True vs Predicted")

plt.legend()
plt.grid()

plt.show()

In [ ]:
plt.figure(figsize=(8,4))

plt.plot(x, df["TRUE_GCV"], label="True GCV", color="orange", linewidth=2)
plt.plot(x, df["PRED_GCV"], label="Pred GCV", color="blue", linewidth=2)

plt.xlabel("Sample Index")
plt.ylabel("GCV")
plt.title("GCV: True vs Predicted")

plt.legend()
plt.grid()

plt.show()